# 02 — Matrices et prétraitements (tâches 11 à 14)


Le split QC est chargé, jamais reconstruit. Les échecs de matrice et de prétraitement sont persistés séparément.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import experiment_config as cfg
from src.io.database_h5 import load_nir_uco_h5
from src.matrices.matrix_registry import build_matrix_output
from src.workflows.matrix_preprocessing import (
    assert_wavelength_lock,
    build_matrix_coverage_table,
    build_wavelength_config,
    evaluate_balanced_sampling_grid,
    evaluate_preprocessing_grid,
    summarize_matrix_output,
)
from src.workflows.protocol_split import eligible_object_ids

QC_DIR = PROJECT_ROOT.joinpath(*cfg.QC_RESULTS_RELATIVE_DIR)
RESULTS_DIR = PROJECT_ROOT / "results" / f"{cfg.MATRIX_RESULTS_DIR_PREFIX}_{cfg.DEFAULT_RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {
    key: RESULTS_DIR / filename for key, filename in cfg.MATRIX_OUTPUT_FILENAMES.items()
}
split_manifest = pd.read_parquet(
    QC_DIR / cfg.QC_OUTPUT_FILENAMES["split_manifest"]
)
object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=True,
)
calibration_ids = eligible_object_ids(split_manifest, "calibration")
validation_ids = eligible_object_ids(split_manifest, "validation")
if not calibration_ids or not validation_ids:
    raise RuntimeError("Calibration and validation must both contain QC-eligible objects.")


In [2]:
wavelength_candidate = build_wavelength_config(
    image_db,
    object_db,
    wavelength_mode=cfg.DEFAULT_WAVELENGTH_MODE,
    protocol_version=cfg.PROTOCOL_VERSION,
    n_remove_start=cfg.N_REMOVE_START,
    n_stop_end=cfg.N_STOP_END,
    window_min_nm=(
        cfg.WAVELENGTH_WINDOW_MIN_NM if cfg.USE_WAVELENGTH_WINDOW else None
    ),
    window_max_nm=(
        cfg.WAVELENGTH_WINDOW_MAX_NM if cfg.USE_WAVELENGTH_WINDOW else None
    ),
)
if OUTPUT["wavelength_config"].exists():
    assert_wavelength_lock(
        pd.read_parquet(OUTPUT["wavelength_config"]),
        wavelength_candidate,
    )
wavelength_candidate.to_parquet(OUTPUT["wavelength_config"], index=False)


In [3]:
matrix_summaries, coverage_tables, matrix_errors = [], [], []
matrix_outputs = {}
for role, ids in (("calibration", calibration_ids), ("validation", validation_ids)):
    for method, strategy, m in cfg.PREPROCESSING_MATRIX_SPECS:
        matrix_id = method
        if strategy is not None:
            matrix_id += f"_{strategy}_m{int(m)}"
        matrix_id = f"{role}_{matrix_id}"
        try:
            output = build_matrix_output(
                object_db,
                matrix_method=method,
                filters={"object_id": ids},
                m=cfg.M_BALANCED_PIXELS if m is None else int(m),
                random_state=cfg.RANDOM_STATE,
                replace=cfg.REPLACE_BALANCED_PIXELS,
                balanced_pixel_strategy="random" if strategy is None else strategy,
                under_m_policy=cfg.BALANCED_SAMPLING_UNDER_M_POLICY,
                require_two_classes=True,
            )
            row, _ = summarize_matrix_output(
                output.X,
                output.y,
                output.metadata,
                method,
                strategy,
                matrix_id=matrix_id,
                protocol_role=role,
                wavelengths=output.wavelengths,
            )
            matrix_summaries.append(row)
            coverage_tables.append(
                build_matrix_coverage_table(output.metadata, matrix_id=matrix_id)
            )
            matrix_outputs[(role, method, strategy, m)] = output
        except Exception as exc:
            matrix_errors.append(
                {
                    "matrix_id": matrix_id,
                    "protocol_role": role,
                    "matrix_method": method,
                    "balanced_pixel_strategy": strategy,
                    "m": m,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )
matrix_summary = pd.DataFrame(
    matrix_summaries, columns=cfg.MATRIX_SUMMARY_REQUIRED_COLUMNS
)
matrix_coverage = (
    pd.concat(coverage_tables, ignore_index=True)
    if coverage_tables else pd.DataFrame(columns=cfg.MATRIX_COVERAGE_COLUMNS)
)
matrix_error_table = pd.DataFrame(
    matrix_errors, columns=cfg.MATRIX_ERROR_COLUMNS
)
matrix_summary.to_parquet(OUTPUT["matrix_summary"], index=False)
matrix_coverage.to_parquet(OUTPUT["matrix_coverage"], index=False)
matrix_error_table.to_parquet(OUTPUT["matrix_errors"], index=False)
if matrix_errors:
    raise RuntimeError(f"Candidate matrix failures: {matrix_errors}")


In [4]:
m_feasibility, pixel_sampling_diagnostics = evaluate_balanced_sampling_grid(
    object_db,
    filters={"object_id": calibration_ids},
    m_values=cfg.BALANCED_SAMPLING_M_VALUES,
    strategies=cfg.BALANCED_PIXEL_STRATEGIES,
    seeds=cfg.BALANCED_SAMPLING_SEEDS,
    replace=cfg.REPLACE_BALANCED_PIXELS,
    under_m_policy=cfg.BALANCED_SAMPLING_UNDER_M_POLICY,
    min_eligible_rate=cfg.BALANCED_SAMPLING_MIN_ELIGIBLE_RATE,
    return_diagnostics=True,
)
m_feasibility.to_parquet(OUTPUT["m_feasibility"], index=False)
pixel_sampling_diagnostics.to_parquet(
    OUTPUT["pixel_sampling_diagnostics"], index=False
)


In [5]:
preprocessing_tables, preprocessing_errors = [], []
for method, strategy, m in cfg.PREPROCESSING_MATRIX_SPECS:
    fit = matrix_outputs[("calibration", method, strategy, m)]
    evaluation = matrix_outputs[("validation", method, strategy, m)]
    pair_id = method if strategy is None else f"{method}_{strategy}_m{int(m)}"
    summary, _, errors = evaluate_preprocessing_grid(
        fit.X,
        evaluation.X,
        preprocessing_configs=cfg.PREPROCESSING_CONFIGS_TO_COMPARE,
        sg_windows=cfg.SG_WINDOW_CHOICES,
        sg_polyorder=cfg.SG_POLYORDER,
        wavelengths=fit.wavelengths,
        matrix_id=pair_id,
        fit_role="calibration",
        eval_role="validation",
    )
    preprocessing_tables.append(summary)
    if not errors.empty:
        preprocessing_errors.append(errors)
preprocessing_validation = pd.concat(preprocessing_tables, ignore_index=True)
preprocessing_error_table = (
    pd.concat(preprocessing_errors, ignore_index=True)
    if preprocessing_errors
    else pd.DataFrame(
        columns=cfg.PREPROCESSING_ERROR_COLUMNS
    )
)
preprocessing_validation.to_parquet(
    OUTPUT["preprocessing_validation"], index=False
)
preprocessing_error_table.to_parquet(
    OUTPUT["preprocessing_errors"], index=False
)
if not preprocessing_error_table.empty:
    print(
        "Some preprocessing candidates failed technical validation; "
        "inspect preprocessing_errors.parquet."
    )
preprocessing_validation.query("status == 'accepted'")


Some preprocessing candidates failed technical validation; inspect preprocessing_errors.parquet.


,matrix_id,fit_role,eval_role,wavelength_axis_id,preprocessing,steps,sg_window_length,sg_polyorder,deriv,status,...,n_features_after,band_count_unchanged,n_nan,n_inf,zero_variance_band_rate,saturation_rate,global_min,global_max,repeatability_error,name_steps_coherent
0,object_mean,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,raw,raw,NaN,NaN,NaN,accepted,...,63.0,True,0.0,0.0,0.0,0.0,0.070981,0.644327,0.0,True
1,object_mean,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance,absorbance,NaN,NaN,NaN,accepted,...,63.0,True,0.0,0.0,0.0,0.0,0.190894,1.148861,0.0,True
2,object_mean,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,snv,snv,NaN,NaN,NaN,accepted,...,63.0,True,0.0,0.0,0.0,0.0,-2.891443,1.288223,0.0,True
3,object_mean,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,msc,msc,NaN,NaN,NaN,accepted,...,63.0,True,0.0,0.0,0.0,0.0,0.094059,0.456861,0.0,True
4,object_mean,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,vector_norm,vector_norm,NaN,NaN,NaN,accepted,...,63.0,True,0.0,0.0,0.0,0.0,0.035887,0.172308,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,snv_sg_d2,snv + sg_d2,7.0,2.0,2.0,accepted,...,63.0,True,0.0,0.0,0.0,0.0,-0.002727,0.001921,0.0,True
531,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,snv_sg_d2,snv + sg_d2,9.0,2.0,2.0,accepted,...,63.0,True,0.0,0.0,0.0,0.0,-0.002023,0.001464,0.0,True
532,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,snv_sg_d2,snv + sg_d2,11.0,2.0,2.0,accepted,...,63.0,True,0.0,0.0,0.0,0.0,-0.001472,0.001118,0.0,True
533,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,snv_sg_d2,snv + sg_d2,13.0,2.0,2.0,accepted,...,63.0,True,0.0,0.0,0.0,0.0,-0.001048,0.000821,0.0,True


In [6]:
preprocessing_error_table

,matrix_id,fit_role,eval_role,wavelength_axis_id,preprocessing,sg_window_length,error_type,error
0,all_pixels,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance,NaN,ValueError,Absorbance is undefined for non-positive refle...
1,all_pixels,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_snv,NaN,ValueError,Absorbance is undefined for non-positive refle...
2,all_pixels,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_msc,NaN,ValueError,Absorbance is undefined for non-positive refle...
3,all_pixels,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_sg_smooth,5.0,ValueError,Absorbance is undefined for non-positive refle...
4,all_pixels,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_sg_smooth,7.0,ValueError,Absorbance is undefined for non-positive refle...
...,...,...,...,...,...,...,...,...
190,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_snv_sg_d2,7.0,ValueError,Absorbance is undefined for non-positive refle...
191,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_snv_sg_d2,9.0,ValueError,Absorbance is undefined for non-positive refle...
192,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_snv_sg_d2,11.0,ValueError,Absorbance is undefined for non-positive refle...
193,balanced_pixels_center_m20,calibration,validation,5277518546b3a1d75330cf8008d875dd51ed1f47c940ff...,absorbance_snv_sg_d2,13.0,ValueError,Absorbance is undefined for non-positive refle...
